# DuckDB: локальная аналитика


Показать отношение в памяти, чтение Parquet, описание и план DuckDB.

Нужны пакеты `duckdb` и `pandas`; примеры в памяти работают без сервисов. Пример с Parquet пропускается, если `EXAMPLE_PARQUET_PATH` не задан.

Ограничения: Когда данные малы, нужен локальный SQL или быстрый прототип без распределённого кластера. Не открывайте большой набор озера данных целиком в памяти одного процесса.


## Основные понятия
DuckDB работает внутри процесса Python. Отношение можно строить лениво, а `show` и `fetchall` материализуют небольшой результат.


In [ ]:
import duckdb
import pandas as pd
import os

## Примеры


**1. Таблица pandas**

Создайте малый набор и сгруппируйте его SQL.



In [ ]:
connection = duckdb.connect()
try:
    matches = pd.DataFrame({
        "match_id": [1, 2, 3, 4], 
        "duration": [1800, 2400, 3600, 1500], 
        "radiant_win": [True, False, True, False]
    })
    connection.sql("""
        select radiant_win, count(*) as matches, round(avg(duration) / 60, 1) as average_minutes 
        from matches 
        group by radiant_win 
        order by radiant_win
    """).show()
finally:
    connection.close()

**2. Отношение**

Соберите цепочку отбора и выбора в памяти.



In [ ]:
connection = duckdb.connect()
try:
    frame = pd.DataFrame({"match_id": [1, 2, 3], "duration": [1800, 2400, 1200]})
    connection.from_df(frame).filter("duration >= 1800").project("match_id, duration / 60 as minutes").show()
finally:
    connection.close()

**3. Parquet**

Прочитайте небольшой файл через параметризованный путь из окружения.



In [ ]:
path = os.getenv("EXAMPLE_PARQUET_PATH")
if not path:
    print("Пример пропущен: EXAMPLE_PARQUET_PATH не задан")
else:
    connection = duckdb.connect()
    try:
        print(connection.execute("""
            select * 
            from read_parquet(?) 
            limit 10
        """, [path]).fetchdf())
    finally:
        connection.close()


**4. Описание**

Получите статистическое описание малого отношения.



In [ ]:
connection = duckdb.connect()
try:
    connection.sql("""
        select * 
        from (values (1, 10.0), (2, 20.0), (3, 15.0)) as data(id, value)
    """).describe().show()
finally:
    connection.close()

**5. План**

Посмотрите план запроса без чтения большого набора.



In [ ]:
connection = duckdb.connect()
try:
    print(connection.sql("""
        explain 
        select id, value 
        from (values (1, 10), (2, 20)) as data(id, value) 
        where value > 10
    """).fetchall()[0][1])
finally:
    connection.close()